# EfficientNet ASPP CBAM Data Check

This notebook prepares the next ASPP-CBAM optimisation attempt by re-checking the dataset distribution first. It intentionally stops after class distribution analysis so that the training strategy can be chosen based on the actual imbalance in the current train/validation/test split.


## 1. Libraries

This section imports only the libraries needed for path handling, JSON loading, mask reading, counting, and table display. The progress bar uses plain text formatting to avoid VS Code widget rendering issues.


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.std import tqdm

print("Libraries loaded.")


Libraries loaded.


## 2. Constants and Paths

This section locates the project root, dataset folder, split JSON, and category file. It supports both the local VS Code workspace and common Colab Drive paths.


In [2]:
CANDIDATE_ROOTS = [
    Path.cwd().resolve(),
    Path("/content/drive/MyDrive/COMP9444/9444"),
    Path("/content/9444"),
]


def find_project_root():
    for root in CANDIDATE_ROOTS:
        data_root = root / "UECFOODPIX" / "data" / "UECFoodPIX"
        split_file = root / "uecfoodpix_split_train3000_val500.json"
        category_file = root / "UECFOODPIX" / "data" / "category.txt"
        if data_root.exists() and split_file.exists() and category_file.exists():
            return root
    checked = "\n".join(str(root) for root in CANDIDATE_ROOTS)
    raise FileNotFoundError("Missing dataset, split JSON, or category file. Checked:\n" + checked)


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "UECFOODPIX" / "data" / "UECFoodPIX"
SPLIT_FILE = PROJECT_ROOT / "uecfoodpix_split_train3000_val500.json"
CATEGORY_FILE = PROJECT_ROOT / "UECFOODPIX" / "data" / "category.txt"

NUM_CLASSES = 103

print("Project root:", PROJECT_ROOT)
print("Data root:", DATA_ROOT)
print("Split file:", SPLIT_FILE)
print("Category file:", CATEGORY_FILE)


Project root: /content/drive/MyDrive/COMP9444/9444
Data root: /content/drive/MyDrive/COMP9444/9444/UECFOODPIX/data/UECFoodPIX
Split file: /content/drive/MyDrive/COMP9444/9444/uecfoodpix_split_train3000_val500.json
Category file: /content/drive/MyDrive/COMP9444/9444/UECFOODPIX/data/category.txt


## 3. Load Split and Class Names

This section loads the fixed train/validation split from JSON, lists the official test ids, and reads the 102 food class names plus background.


In [3]:
def load_split(split_file):
    split_data = json.loads(Path(split_file).read_text(encoding="utf-8"))
    train_ids = [str(item) for item in split_data["train_ids"]]
    val_ids = [str(item) for item in split_data["val_ids"]]

    assert len(train_ids) == split_data["train_size"]
    assert len(val_ids) == split_data["val_size"]
    assert len(train_ids) == len(set(train_ids))
    assert len(val_ids) == len(set(val_ids))
    assert not (set(train_ids) & set(val_ids))
    return train_ids, val_ids, split_data


def list_test_ids(data_root):
    return sorted(path.stem for path in (data_root / "test" / "img").glob("*.jpg"))


def load_class_names(category_file, num_classes=NUM_CLASSES):
    class_names = [f"class_{index}" for index in range(num_classes)]
    class_names[0] = "background"

    with open(category_file, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if not line or line.lower().startswith("id"):
                continue
            parts = line.split(maxsplit=1)
            if len(parts) != 2:
                continue
            class_id = int(parts[0])
            if 0 <= class_id < num_classes:
                class_names[class_id] = parts[1]
    return class_names


train_ids, val_ids, split_data = load_split(SPLIT_FILE)
test_ids = list_test_ids(DATA_ROOT)
CLASS_NAMES = load_class_names(CATEGORY_FILE)

print(f"Train images: {len(train_ids)}")
print(f"Validation images: {len(val_ids)}")
print(f"Test images: {len(test_ids)}")
print("First classes:", CLASS_NAMES[:10])


Train images: 3000
Validation images: 500
Test images: 1000
First classes: ['background', 'rice', 'eels on rice', 'pilaf', "chicken-'n'-egg on rice", 'pork cutlet on rice', 'beef curry', 'sushi', 'chicken rice', 'fried rice']


## 4. Class Distribution from Current Split JSON

This cell counts every class in the current split. For each class, it reports how many images contain that class and how many mask pixels belong to that class in train, validation, and test. This is the key checkpoint before deciding whether the next ASPP-CBAM version should use class-balanced sampling, stronger class weights, focal loss, or category grouping.


In [4]:
def mask_path_for_id(data_root, split, image_id):
    return data_root / split / "mask" / f"{image_id}.png"


def count_classes_for_ids(data_root, split, image_ids, num_classes=NUM_CLASSES):
    image_counts = np.zeros(num_classes, dtype=np.int64)
    pixel_counts = np.zeros(num_classes, dtype=np.int64)

    for image_id in tqdm(
        image_ids,
        desc=f"Counting {split}",
        bar_format="{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]",
    ):
        mask = Image.open(mask_path_for_id(data_root, split, image_id))
        if mask.mode != "L":
            mask = mask.split()[0]
        mask_array = np.asarray(mask, dtype=np.int64)
        mask_array = mask_array[(0 <= mask_array) & (mask_array < num_classes)]

        labels, counts = np.unique(mask_array, return_counts=True)
        image_counts[labels] += 1
        pixel_counts[labels] += counts

    return image_counts, pixel_counts


train_image_counts, train_pixel_counts = count_classes_for_ids(DATA_ROOT, "train", train_ids)
val_image_counts, val_pixel_counts = count_classes_for_ids(DATA_ROOT, "train", val_ids)
test_image_counts, test_pixel_counts = count_classes_for_ids(DATA_ROOT, "test", test_ids)

distribution = pd.DataFrame({
    "class_id": np.arange(NUM_CLASSES),
    "class_name": CLASS_NAMES,
    "train_images": train_image_counts,
    "val_images": val_image_counts,
    "test_images": test_image_counts,
    "train_pixels": train_pixel_counts,
    "val_pixels": val_pixel_counts,
    "test_pixels": test_pixel_counts,
})

for split_name in ["train", "val", "test"]:
    total_pixels = max(int(distribution[f"{split_name}_pixels"].sum()), 1)
    distribution[f"{split_name}_pixel_percent"] = distribution[f"{split_name}_pixels"] / total_pixels * 100

distribution["train_val_image_gap"] = distribution["train_images"] - distribution["val_images"]
distribution["has_train"] = distribution["train_images"] > 0
distribution["has_val"] = distribution["val_images"] > 0
distribution["has_test"] = distribution["test_images"] > 0

pd.set_option("display.max_rows", NUM_CLASSES)
pd.set_option("display.max_columns", None)
display(distribution)

print("Foreground classes missing from train:", distribution.loc[(distribution["class_id"] != 0) & ~distribution["has_train"], ["class_id", "class_name"]].to_dict("records"))
print("Foreground classes missing from validation:", distribution.loc[(distribution["class_id"] != 0) & ~distribution["has_val"], ["class_id", "class_name"]].to_dict("records"))
print("Foreground classes missing from test:", distribution.loc[(distribution["class_id"] != 0) & ~distribution["has_test"], ["class_id", "class_name"]].to_dict("records"))

print("\n10 rarest foreground classes in train by image count:")
display(distribution[distribution["class_id"] != 0].sort_values(["train_images", "train_pixels"]).head(10))

print("\n10 most frequent foreground classes in train by image count:")
display(distribution[distribution["class_id"] != 0].sort_values(["train_images", "train_pixels"], ascending=False).head(10))


Counting test: 100%|██████████| 1000/1000 [00:05<00:00, 193.43it/s]


,class_id,class_name,train_images,val_images,test_images,train_pixels,val_pixels,test_pixels,train_pixel_percent,val_pixel_percent,test_pixel_percent,train_val_image_gap,has_train,has_val,has_test
0,0,background,2936,487,999,317825360,50120010,99844155,58.464942,59.202018,58.297400,2449,True,True,True
1,1,rice,144,28,77,6007562,988317,1875116,1.105109,1.167405,1.094850,116,True,True,True
2,2,eels on rice,28,5,10,1454256,244382,551135,0.267515,0.288665,0.321799,23,True,True,True
3,3,pilaf,29,5,8,1477512,537016,293736,0.271793,0.634326,0.171508,24,True,True,True
4,4,chicken-'n'-egg on rice,29,5,8,1891747,112903,506325,0.347993,0.133362,0.295635,24,True,True,True
5,5,pork cutlet on rice,37,6,10,1979243,338429,425099,0.364088,0.399754,0.248208,31,True,True,True
6,6,beef curry,58,12,19,3139319,697199,999055,0.577487,0.823535,0.583332,46,True,True,True
7,7,sushi,35,6,10,1056155,125654,893947,0.194283,0.148423,0.521961,29,True,True,True
8,8,chicken rice,23,5,6,2153600,338315,421160,0.396161,0.399619,0.245909,18,True,True,True
9,9,fried rice,37,6,12,2746056,468113,721474,0.505145,0.552938,0.421257,31,True,True,True


Foreground classes missing from train: []
Foreground classes missing from validation: []
Foreground classes missing from test: []

10 rarest foreground classes in train by image count:


,class_id,class_name,train_images,val_images,test_images,train_pixels,val_pixels,test_pixels,train_pixel_percent,val_pixel_percent,test_pixel_percent,train_val_image_gap,has_train,has_val,has_test
81,81,steamed meat dumpling,18,3,9,589161,29322,211720,0.108378,0.034635,0.123620,15,True,True,True
14,14,roll bread,21,3,12,856028,71955,592778,0.157469,0.084994,0.346114,18,True,True,True
33,33,grilled eggplant,21,3,12,1813236,349667,905041,0.333550,0.413028,0.528439,18,True,True,True
46,46,grilled salmon,22,3,15,613391,52381,172994,0.112835,0.061873,0.101008,19,True,True,True
61,61,beef steak,22,4,6,931023,118396,198995,0.171265,0.139850,0.116190,18,True,True,True
71,71,egg roll,23,4,7,742536,86643,182486,0.136592,0.102343,0.106551,19,True,True,True
60,60,hambarg steak,23,6,10,1054750,93856,314818,0.194024,0.110863,0.183817,17,True,True,True
41,41,ganmodoki,23,3,12,1128132,109904,757969,0.207523,0.129819,0.442566,20,True,True,True
93,93,kinpira-style sauteed burdock,23,5,10,1724610,234009,348708,0.317247,0.276413,0.203605,18,True,True,True
34,34,sauteed spinach,23,3,9,1864244,196470,536034,0.342933,0.232071,0.312982,20,True,True,True



10 most frequent foreground classes in train by image count:


,class_id,class_name,train_images,val_images,test_images,train_pixels,val_pixels,test_pixels,train_pixel_percent,val_pixel_percent,test_pixel_percent,train_val_image_gap,has_train,has_val,has_test
101,101,others,673,124,303,17574386,2431232,5572278,3.232862,2.871784,3.253564,549,True,True,True
102,102,beverage,160,28,91,1781892,261292,865647,0.327784,0.308639,0.505437,132,True,True,True
36,36,miso soup,149,27,82,4407909,773743,1888819,0.810848,0.913949,1.102851,122,True,True,True
1,1,rice,144,28,77,6007562,988317,1875116,1.105109,1.167405,1.094850,116,True,True,True
87,87,green salad,94,14,91,3628422,396413,2491427,0.667459,0.468245,1.454704,80,True,True,True
23,23,ramen noodle,84,14,35,7984496,1464274,2802009,1.468772,1.729608,1.636048,70,True,True,True
6,6,beef curry,58,12,19,3139319,697199,999055,0.577487,0.823535,0.583332,46,True,True,True
68,68,egg sunny-side up,55,9,19,2176994,600522,819069,0.400465,0.709340,0.478241,46,True,True,True
17,17,hamburger,54,8,25,4051406,582302,1419522,0.745268,0.687818,0.828836,46,True,True,True
12,12,toast,53,8,23,3178474,603690,830980,0.584690,0.713082,0.485196,45,True,True,True


## Next Step Placeholder

！！！请先查看上面的类别分布结果，再决定下一步数据重处理和 ASPP-CBAM 训练策略！！！
